<a href="https://colab.research.google.com/github/javierrcastroo/DL_Proyecto/blob/main/NLP_Pr%C3%A1ctica_1_Preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import nltk
from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize, sent_tokenize

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
nltk.download('punkt_tab')

from bs4 import BeautifulSoup

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Preprocesamiento

Este preprocesamiento se va a realizar usando principalmente la librería NLTK.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving exploracion_anotado.csv to exploracion_anotado.csv


In [ ]:
import pandas as pd

df = pd.read_csv("exploracion_anotado.csv")

## Tokenización

Vamos a aplicar la tokenización a nuestro dataset de ejemplo:

In [ ]:
df['texto_tokenizado'] = df['text'].apply(lambda x: word_tokenize(x))
df[['text', 'texto_tokenizado']].head()

,text,texto_tokenizado
0,is a historical account written in the early 2...,"[is, a, historical, account, written, in, the,..."
1,is an English translation of the Bible publish...,"[is, an, English, translation, of, the, Bible,..."
2,is a memoir and travel book published in 1883....,"[is, a, memoir, and, travel, book, published, ..."
3,is a historical account that was written in th...,"[is, a, historical, account, that, was, writte..."
4,is a historical account written in the late 19...,"[is, a, historical, account, written, in, the,..."


A nivel de oraciones, NLTK ofrece el método sent_tokenize:

In [ ]:
df['texto_tokenizado_oraciones'] = df['text'].apply(lambda x: sent_tokenize(x))
df[['text', 'texto_tokenizado_oraciones']].head()

,text,texto_tokenizado_oraciones
0,is a historical account written in the early 2...,[is a historical account written in the early ...
1,is an English translation of the Bible publish...,[is an English translation of the Bible publis...
2,is a memoir and travel book published in 1883....,[is a memoir and travel book published in 1883...
3,is a historical account that was written in th...,[is a historical account that was written in t...
4,is a historical account written in the late 19...,[is a historical account written in the late 1...


## Simplificación

### Ruido

#### Eliminanción de texto redundante

En cada uno de los textos tenemos '...' a mitad del texto, que se debe a una lectura literal del HTML del web scrapping.

In [ ]:
import re

def tiene_tres_puntos(texto):
    if not isinstance(texto, str):
        return False
    return bool(re.search(r'\.{3,}', texto))

filas_con_puntos = df["text"].apply(tiene_tres_puntos).sum()

if filas_con_puntos == df.shape[0]:
    print("Todos los textos tienen los tres puntos")
else:
    print(f"No todos los textos tienen los tres puntos. Solo {filas_con_puntos} de {df.shape[0]}.")


Todos los textos tienen los tres puntos


Tras la comprobación, hemos podido ver que todos nuestros textos cuentan con esta limitación, por lo que se tiene que hacer una eliminación de puntuaciones.

#### Eliminación de puntuaciones

In [ ]:
df["texto_sin_puntuacion"] = df["text"].str.replace(r"[^\w\s]+", "", regex=True)

pd.set_option('display.max_colwidth', None)
df[['text', 'texto_sin_puntuacion']].head()

,text,texto_sin_puntuacion
0,"is a historical account written in the early 20th century. The book serves as a comprehensive roster for the 42nd Infantry Division, known as the Rainbow Division, detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that the Rainbow Division represents units from twenty-six states across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse backgrounds of its members.",is a historical account written in the early 20th century The book serves as a comprehensive roster for the 42nd Infantry Division known as the Rainbow Division detailing the officers and soldiers who served within it The roster provides a glimpse into the composition and leadership of a significant unit in the US military during World War I The opening of the work introduces Lieutenant Harold Stanley Johnsons foreword highlighting the formation and purpose of the roster He emphasizes the importance of personal connections among soldiers for effective teamwork noting that the Rainbow Division represents units from twentysix states across the US Johnson acknowledges the pride and honor of serving in this unit especially as they prepare to be among the first American forces deployed to Europe in the Great War The beginning also outlines notable figures within the division including MajorGeneral Wm A Mann and Colonel Douglass MacArthur alongside illustrating the diverse backgrounds of its members
1,"is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today.",is an English translation of the Bible published between 1582 and 1610 Created by English Catholic scholars in exile during the Reformation this translation rendered the Latin Vulgate into Early Modern English as a CounterReformation effort The New Testament appeared in Rheims in 1582 while the Old Testament followed in Douai nearly three decades later Later revised by Bishop Richard Challoner in the mid1700s this translation influenced the King James Version and remains significant for traditional Englishspeaking Catholics today
2,"is a memoir and travel book published in 1883. It recounts Twain's experiences as a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicles his return journey decades later, observing how railroads, growing cities, and time have transformed the river and its culture. Blending personal history with tall tales and ... social commentary, Twain captures a vanishing era of American river life.",is a memoir and travel book published in 1883 It recounts Twains experiences as a young steamboat pilots apprentice on the Mississippi River before the Civil War detailing the art of navigating the everchanging waters The second half chronicles his return journey decades later observing how railroads growing cities and time ha

#### Eliminación de stopwords

Las stopwords o palabras vacías, son palabras que no tienen significado como artículos, pronombres, preposiciones, etc. En muchos procesos de PLN son eliminadas ya que no aportan más que mayor complejidad.

In [ ]:
stopwords = set(stopwords.words('english'))
print(len(stopwords))
print(stopwords)

198
{'ma', 'nor', 'll', 'ours', 'very', 'at', 'through', "we'll", 'few', 'for', 'those', "mustn't", "i've", 'most', 'during', "i'd", 'all', "they've", 'own', 't', 'has', "shouldn't", "mightn't", 'them', 'than', 'are', 'other', "don't", "it'll", 'this', 'which', 'won', 'him', 'aren', 'mightn', 'shouldn', 'because', 'doing', 'and', 'do', "he'll", 'so', 'don', 'doesn', "shan't", "we've", 'our', 'when', 'yours', 'i', "isn't", 'ain', 'hadn', 'mustn', 'had', 'should', 'they', "wasn't", 'with', "she's", 'we', 'too', "haven't", 'if', 'his', 'can', 'itself', 'theirs', 'then', "won't", "you'd", 'is', "he'd", "doesn't", 'hers', "we'd", "couldn't", 'm', 'of', 'wasn', 'been', 's', 'once', 'you', 'does', 'yourself', 'her', "didn't", 'on', 'further', 'only', 'that', 'were', "wouldn't", "she'll", 'until', 'not', "i'll", 'a', 'under', 'into', 'no', 'your', 'in', 'how', 'myself', 'from', 'o', 'here', 'up', 'have', 'hasn', 'weren', "hadn't", 'down', 're', "they'd", 'she', 'before', 'each', "it'd", 'would

In [ ]:
def quitar_stopwords(texto, stopwords):
    palabras = texto.split()
    palabras_filtradas = [p for p in palabras if p.lower() not in stopwords]
    # convertimos a minúscula para realizar la comparación
    return " ".join(palabras_filtradas)

pd.set_option('display.max_colwidth', None)
df["texto_sin_stopwords"] = df["text"].apply(lambda x: quitar_stopwords(x, stopwords))
df[['text','texto_sin_stopwords']].head()

,text,texto_sin_stopwords
0,"is a historical account written in the early 20th century. The book serves as a comprehensive roster for the 42nd Infantry Division, known as the Rainbow Division, detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that the Rainbow Division represents units from twenty-six states across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse backgrounds of its members.","historical account written early 20th century. book serves comprehensive roster 42nd Infantry Division, known Rainbow Division, detailing officers soldiers served within it. roster provides glimpse composition leadership significant unit U.S. military World War I. opening ... work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting formation purpose roster. emphasizes importance personal connections among soldiers effective teamwork, noting Rainbow Division represents units twenty-six states across U.S. Johnson acknowledges pride honor serving unit, especially prepare among first American forces deployed Europe Great War. beginning also outlines notable figures within division, including Major-General Wm. A. Mann Colonel Douglass MacArthur, alongside illustrating diverse backgrounds members."
1,"is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today.","English translation Bible published 1582 1610. Created English Catholic scholars exile Reformation, translation rendered Latin Vulgate Early Modern English Counter-Reformation effort. New Testament appeared Rheims 1582, Old Testament followed Douai nearly three decades later. Later revised Bishop Richard Challoner mid-1700s, translation influenced ... King James Version remains significant traditional English-speaking Catholics today."
2,"is a memoir and travel book published in 1883. It recounts Twain's experiences as a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicles his return journey decades later, observing how railroads, growing cities, and time have transformed the river and its culture. Blending personal history with tall tales and ... social commentary, Twain captures a vanishing era of American river life.","memoir travel book published 1883. recounts Twain's experiences young steamboat pilot's apprentice Mississippi River Civil War, detailing art navigating ever-changing waters. second half chronicles return journey decades later, observing railroads, growing cities, time transformed river culture. Blending personal history tall tales ... social commentary, Twain captures vanishing era American river life."
3,"is a historical account that was written in the early 20th century. The text provides a firsthand exploration of Beresford's life and career from his entry into the Royal Navy in 1859 to his retirem

#### Eliminación de HTMLs

In [ ]:
def quitar_html(texto):
    return BeautifulSoup(texto, "html.parser").get_text()

df["texto_sin_html"] = df["text"].apply(quitar_html)
pd.set_option('display.max_colwidth', None)
df[["text","texto_sin_html"]].head()


,text,texto_sin_html
0,"is a historical account written in the early 20th century. The book serves as a comprehensive roster for the 42nd Infantry Division, known as the Rainbow Division, detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that the Rainbow Division represents units from twenty-six states across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse backgrounds of its members.","is a historical account written in the early 20th century. The book serves as a comprehensive roster for the 42nd Infantry Division, known as the Rainbow Division, detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that the Rainbow Division represents units from twenty-six states across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse backgrounds of its members."
1,"is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today.","is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today."
2,"is a memoir and travel book published in 1883. It recounts Twain's experiences as a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicles his return journey decades later, observing how railroads, growing cities, and time have transformed the river and its culture. Blending personal history with tall tales and ... social commentary, Twain captures a vanishing era of American river life.","is a memoir and travel book published in 1883. It recounts Twain's experiences as a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicles his return journey decades later, obse

###  Normalización

La lematización es un proceso lingüístico que consiste en, dada una forma flexionada, obtener el lema correspondiente.

El lema es la forma que por convenio se acepta como representante de todas las formas flexionadas de una misma palabra

Stemming es un método para reducir una palabra a su raíz.

Normalmente se elige una de las dos, pero no ambas.

#### Lematizar

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
lemmatizer = WordNetLemmatizer()

def lematizar_texto(texto):
    palabras = texto.split()
    # Lematizamos cada palabra
    palabras_lematizadas = [lemmatizer.lemmatize(p) for p in palabras]
    return " ".join(palabras_lematizadas)

pd.set_option('display.max_colwidth', None)
df["texto_lematizado"] = df["text"].apply(lematizar_texto)
df[['text', 'texto_lematizado']].head()

,text,texto_lematizado
0,"is a historical account written in the early 20th century. The book serves as a comprehensive roster for the 42nd Infantry Division, known as the Rainbow Division, detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that the Rainbow Division represents units from twenty-six states across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse backgrounds of its members.","is a historical account written in the early 20th century. The book serf a a comprehensive roster for the 42nd Infantry Division, known a the Rainbow Division, detailing the officer and soldier who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connection among soldier for effective teamwork, noting that the Rainbow Division represents unit from twenty-six state across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially a they prepare to be among the first American force deployed to Europe in the Great War. The beginning also outline notable figure within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse background of it members."
1,"is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today.","is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholar in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English a a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decade later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today."
2,"is a memoir and travel book published in 1883. It recounts Twain's experiences as a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicles his return journey decades later, observing how railroads, growing cities, and time have transformed the river and its culture. Blending personal history with tall tales and ... social commentary, Twain captures a vanishing era of American river life.","is a memoir and travel book published in 1883. It recount Twain's experience a a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicle his return journey decade later, observing how railroads, g

Vemos que funciona en varios casos, como por ejemplo que las palabras plurales se lematizan a su versión en singular.

No obstante, este enfoque tiene un problema, que readica sobretodo en el análisis de los verbos: WordNetLemmatizer por defecto cree que todas las palabras son sustantivos. Entonces por ejemplo el verbo "written" del primer texto lo deja intacto, cuando debería de ser "write".

Para conseguirlo, necesitamos etiquetar las palabras con su categoría gramatical antes de lematizarlas (POS tagging). Así, el lematizador sabrá si una palabra actúa como verbo, sustantivo, adjetivo, etc.

In [ ]:
from nltk.corpus import wordnet
nltk.download('averaged_perceptron_tagger_eng')

def obtener_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN # Sustantivo por defecto

def lematizar_con_pos(texto):
    palabras = word_tokenize(texto)
    # Obtenemos las etiquetas POS
    pos_tags = nltk.pos_tag(palabras)

    # Lematizamos usando la etiqueta correcta
    palabras_lematizadas = [
        lemmatizer.lemmatize(palabra, obtener_wordnet_pos(tag))
        for palabra, tag in pos_tags
    ]
    return " ".join(palabras_lematizadas)

df["texto_lematizado_pos"] = df["text"].apply(lematizar_con_pos)
df[['text', 'texto_lematizado', 'texto_lematizado_pos']].head()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


,text,texto_lematizado,texto_lematizado_pos
0,"is a historical account written in the early 20th century. The book serves as a comprehensive roster for the 42nd Infantry Division, known as the Rainbow Division, detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that the Rainbow Division represents units from twenty-six states across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse backgrounds of its members.","is a historical account written in the early 20th century. The book serf a a comprehensive roster for the 42nd Infantry Division, known a the Rainbow Division, detailing the officer and soldier who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connection among soldier for effective teamwork, noting that the Rainbow Division represents unit from twenty-six state across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially a they prepare to be among the first American force deployed to Europe in the Great War. The beginning also outline notable figure within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse background of it members.","be a historical account write in the early 20th century . The book serve a a comprehensive roster for the 42nd Infantry Division , know a the Rainbow Division , detail the officer and soldier who serve within it . The roster provide a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I . The opening ... of the work introduces Lieutenant Harold Stanley Johnson 's foreword , highlight the formation and purpose of the roster . He emphasize the importance of personal connection among soldier for effective teamwork , note that the Rainbow Division represent unit from twenty-six state across the U.S. Johnson acknowledge the pride and honor of serve in this unit , especially a they prepare to be among the first American force deploy to Europe in the Great War . The beginning also outline notable figure within the division , include Major-General Wm . A. Mann and Colonel Douglass MacArthur , alongside illustrate the diverse background of it member ."
1,"is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today.","is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholar in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English a a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Doua

Podemos observar que en el primer texto, al lematizarlo con el POS tagging, el ejemplo de "written" se lematiza a "write", lo que es completamente cierto y mucho más preciso.

Ahora vamos a ver a modo de ilustración que palabras varían y cuales no en función de la lematización que usemos, con respecto al texto original.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet

texto1 = df['text'].iloc[0]
tokens = word_tokenize(texto1)
pos_tags = nltk.pos_tag(tokens)

encontrados = {'Sustantivo': [], 'Adjetivo': [], 'Verbo': []}

for palabra, tag in pos_tags:
    wn_pos = obtener_wordnet_pos(tag)

    if wn_pos == wordnet.NOUN and tag.startswith('N') and len(encontrados['Sustantivo']) < 3:
        encontrados['Sustantivo'].append((palabra, tag))
    elif wn_pos == wordnet.ADJ and len(encontrados['Adjetivo']) < 3:
        encontrados['Adjetivo'].append((palabra, tag))
    elif wn_pos == wordnet.VERB and len(encontrados['Verbo']) < 3:
        encontrados['Verbo'].append((palabra, tag))

    if all(len(lista) == 3 for lista in encontrados.values()):
        break

print("Comparación de lematización (3 ejemplos por categoría):\n")
for tipo, lista_ejemplos in encontrados.items():
    print(f"=== CATEGORÍA: {tipo} ===")
    for palabra, tag in lista_ejemplos:
        lem_basico = lemmatizer.lemmatize(palabra)
        lem_pos = lemmatizer.lemmatize(palabra, obtener_wordnet_pos(tag))

        print(f"Palabra: {palabra} (Etiqueta: {tag})")
        print(f"  Básica: {lem_basico} | Con POS: {lem_pos}")
    print("\n")


Comparación de lematización (3 ejemplos por categoría):

=== CATEGORÍA: Sustantivo ===
Palabra: account (Etiqueta: NN)
  Básica: account | Con POS: account
Palabra: century (Etiqueta: NN)
  Básica: century | Con POS: century
Palabra: book (Etiqueta: NN)
  Básica: book | Con POS: book


=== CATEGORÍA: Adjetivo ===
Palabra: historical (Etiqueta: JJ)
  Básica: historical | Con POS: historical
Palabra: early (Etiqueta: JJ)
  Básica: early | Con POS: early
Palabra: 20th (Etiqueta: JJ)
  Básica: 20th | Con POS: 20th


=== CATEGORÍA: Verbo ===
Palabra: is (Etiqueta: VBZ)
  Básica: is | Con POS: be
Palabra: written (Etiqueta: VBN)
  Básica: written | Con POS: write
Palabra: serves (Etiqueta: VBZ)
  Básica: serf | Con POS: serve




Podemos observar que al nueva lematización actúa sobretodo en los verbos, lematizandolos de una manera mucho más efectiva y general que en la lematización simple.

Por ejemplo en "serves" el lematizador simple se equivoca y lo trata como un sustantivo, pero el lematizador POS tagging trata su lema como verbo "serve" tras identificarlo.

###  Anonimización

La anonimización sirve para proteger la privacidad eliminando o sustituyendo datos sensibles (nombres, teléfonos, emails) por etiquetas genéricas, evitando que se identifique a personas reales en el dataset.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def anonimizar(texto):
    doc = nlp(texto)
    nuevo_texto = texto
    for ent in doc.ents:
        if ent.label_ in ["PERSON", "ORG"]:
            nuevo_texto = nuevo_texto.replace(ent.text, f"[{ent.label_}]")
    return nuevo_texto

df["text_anonimo"] = df["text"].apply(anonimizar)
df[['text', 'text_anonimo']].head()

,text,text_anonimo
0,"is a historical account written in the early 20th century. The book serves as a comprehensive roster for the 42nd Infantry Division, known as the Rainbow Division, detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant Harold Stanley Johnson's foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that the Rainbow Division represents units from twenty-six states across the U.S. Johnson acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including Major-General Wm. A. Mann and Colonel Douglass MacArthur, alongside illustrating the diverse backgrounds of its members.","is a historical account written in the early 20th century. The book serves as a comprehensive roster for [ORG], known as [ORG], detailing the officers and soldiers who served within it. The roster provides a glimpse into the composition and leadership of a significant unit in the U.S. military during World War I. The opening ... of the work introduces Lieutenant [PERSON] foreword, highlighting the formation and purpose of the roster. He emphasizes the importance of personal connections among soldiers for effective teamwork, noting that [ORG] represents units from twenty-six states across the U.S. [PERSON] acknowledges the pride and honor of serving in this unit, especially as they prepare to be among the first American forces deployed to Europe in the Great War. The beginning also outlines notable figures within the division, including [ORG]. [PERSON] and Colonel [PERSON], alongside illustrating the diverse backgrounds of its members."
1,"is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. The New Testament appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by Bishop Richard Challoner in the mid-1700s, this translation influenced the ... King James Version and remains significant for traditional English-speaking Catholics today.","is an English translation of the Bible published between 1582 and 1610. Created by English Catholic scholars in exile during the Reformation, this translation rendered the Latin Vulgate into Early Modern English as a Counter-Reformation effort. [ORG] appeared in Rheims in 1582, while the Old Testament followed in Douai nearly three decades later. Later revised by [ORG] in the mid-1700s, this translation influenced the ... King [PERSON] and remains significant for traditional English-speaking Catholics today."
2,"is a memoir and travel book published in 1883. It recounts Twain's experiences as a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicles his return journey decades later, observing how railroads, growing cities, and time have transformed the river and its culture. Blending personal history with tall tales and ... social commentary, Twain captures a vanishing era of American river life.","is a memoir and travel book published in 1883. It recounts [ORG]'s experiences as a young steamboat pilot's apprentice on the Mississippi River before the Civil War, detailing the art of navigating the ever-changing waters. The second half chronicles his return journey decades later, observing how railroads, growing cities, and time have transformed the river and its culture. Blending personal history with tal